In [ ]:
import RMF
import numpy as np


def _add_nodes(node, tf, type_prefixes, depth=0):
    '''
    node - rmf node to scan
    tf - typed factory
    type_prefixes - list of full type prefixes (e.g. "Nup1" for "Nup1N")

    adds only nodes whose type name begins with any of the specified type prefixes
    '''
    children = node.get_children()
    ret = []
    if len(children)==0:
        return ret
    if tf.get_is(children[0]):
        child_type = tf.get(children[0]).get_type_name()
        if any([child_type.startswith(tp) for tp in type_prefixes]):
            ret.append(children)
    for c in children:
        ret += _add_nodes(c, tf,  type_prefixes, depth+1)
    return ret

def load_NTR_data(rmf_path, NTR_amount, frames_per_file=1):
    trajectories = np.zeros(shape=[NTR_amount, 3, frames_per_file])

    in_fh = RMF.open_rmf_file_read_only(rmf_path)
    rff = RMF.ReferenceFrameFactory(in_fh)
    tf = RMF.TypedFactory(in_fh)
    NTR_types = [f"NTR14"] # name defined in config.pb

    # load data
    type2chains={}
    for NTR_type in NTR_types:
        type2chains[NTR_type] = _add_nodes(in_fh.get_root_node(), tf, [NTR_type])
    frame_i = 0
    for f_id, f in enumerate(in_fh.get_frames()):
        in_fh.set_current_frame(f)

        # read data
        for NTR_i in range(NTR_amount):
            coord = rff.get(type2chains["NTR14"][0][NTR_i]).get_translation()
            
            trajectories[NTR_i, 0, frame_i] = coord[0] / 10 # In nm
            trajectories[NTR_i, 1, frame_i] = coord[1] / 10
            trajectories[NTR_i, 2, frame_i] = coord[2] / 10
        if frame_i >= frames_per_file:
            break
        frame_i += 1
        
    return trajectories

trajectories = load_NTR_data("/cs/usr/roi.eliasian/LabFolder/Master/NPC-markov/testing/500.movie.rmf", 250, 104)
pass